![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx, and `ibm/granite-4-h-micro` to analyze sentiments of legal documents

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support of sentiment analysis in watsonx. It introduces commands for data retrieval and model testing.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to use `ibm/granite-4-h-micro` model to analyze sentiments of legal documents.

## Use case & dataset

One of the key use cases of legal sentiment analysis is in assisting legal professionals in predicting case outcomes. By analyzing the sentiment expressed in previous court decisions and related documents, sentiment analysis algorithms can identify patterns and correlations between the sentiment and the final verdict. This can help lawyers and judges in assessing the strength of legal arguments, evaluating the potential impact of public opinion on the case, and making more accurate predictions about the likely outcome of ongoing cases.
The dataset consists of two colums; the phrases and the sentiments.

## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Data loading](#Data-loading)
3. [Foundation Models on IBM watsonx.ai](#Foundation-Models-on-IBM-watsonx.ai)
4. [Find legal documents sentiments](#Find-legal-documents-sentiments)
5. [Score the model](#Score-the-model)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U "scikit-learn==1.6.1" | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [4]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id)

<a id="Data-loading"></a>
## Data loading

Download the `legal documents` dataset

In [6]:
import wget

filename = "Legal_Sentences.csv"
url = "https://raw.githubusercontent.com/kmokht1/Datasets/main/Legal_Sentences.csv"

if not os.path.isfile(filename):
    wget.download(url, out=filename)

Read the data

In [7]:
import pandas as pd

data = pd.read_csv("Legal_Sentences.csv", index_col=0)
data = data[["Phrase", "Sentiment"]]
data.head()

,Phrase,Sentiment
0,Getting nowhere with surplusage,-1
1,But the Court nowhere suggested that it would ...,-1
2,Petitioners objection to shaving his beard cla...,-1
3,That result clashes with everything else,-1
4,the tolerable duration of police inquiries in ...,0


Replace numeric sentiment values with text labels

In [8]:
label_map = {-1: "negative", 0: "neutral", 1: "positive"}
data["Sentiment"] = data["Sentiment"].replace(label_map)

Inspect data sample

In [9]:
data.value_counts(["Sentiment"])

Sentiment
negative     282
positive     172
neutral      122
Name: count, dtype: int64

Split the data into training and test sets.

In [10]:
from sklearn.model_selection import train_test_split

data_train, data_test, y_train, y_test = train_test_split(
    data["Phrase"],
    data["Sentiment"],
    test_size=0.3,
    random_state=33,
    stratify=data["Sentiment"],
)
data_train = pd.DataFrame(data_train)
data_test = pd.DataFrame(data_test)

<a id="Foundation-Models-on-IBM-watsonx.ai"></a>
## Foundation Models on IBM watsonx.ai

#### List available models

In [11]:
for model in client.foundation_models.ChatModels:
    print(f"- {model}")

- ibm/granite-4-h-micro
- ibm/ibm-defense-3-3-8b-instruct
- magistral-small-2509
- meta-llama/llama-3-2-1b-instruct
- ministral-8b-instruct-2512
- mistralai/mistral-small-3-2-24b-instruct-2506


You need to specify `model_id` that will be used for inferencing:

In [12]:
model_id = client.foundation_models.ChatModels.GRANITE_4_H_MICRO

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [13]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames as GenParams

parameters = {
    GenParams.TEMPERATURE: 0,
    GenParams.REPETITION_PENALTY: 1,
}

### Initialize the model
Initialize the `ModelInference` class with previous set params.

In [14]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(model_id=model_id, params=parameters, api_client=client)

### Model's details

In [15]:
model.get_details()

{'model_id': 'ibm/granite-4-h-micro',
 'label': 'granite-4-h-micro',
 'provider': 'IBM',
 'source': 'IBM',
 'functions': [{'id': 'text_chat'}],
 'short_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.',
 'long_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instruct models feature improved instruction following (IF) and tool-calling capabilities, making them more effective in enterprise applications.'

<a id="Find-legal-documents-sentiments"></a>
## Find legal documents sentiments

Define instructions for the model. 

In [16]:
def get_messages(
    example_sentences: list[str], example_sentiments: list[str], sentence_to_check: str
) -> list[dict[str, str]]:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a sentiment classification model. "
                "Valid outputs are: positive, neutral, negative. "
                "Output exactly one of these words, which best matches the sentiment of the sentence. "
                "Do not provide an explanation."
            ),
        }
    ]

    for sentence, sentiment in zip(example_sentences, example_sentiments):
        messages.append({"role": "user", "content": sentence})
        messages.append({"role": "assistant", "content": sentiment})

    messages.append({"role": "user", "content": sentence_to_check})

    return messages

Prepare model inputs for few-shot examples.

In [17]:
data_train_and_labels = data_train.copy()
data_train_and_labels["Sentiment"] = y_train

In [18]:
example_entries = data_train_and_labels.groupby("Sentiment").first()

example_sentences: list[str] = []
example_sentiments: list[str] = []

for sentiment, sentence in sorted(example_entries["Phrase"].items()):
    example_sentences.append(sentence)
    example_sentiments.append(str(sentiment))

print(example_sentiments)
print(*example_sentences, sep="\n")

['negative', 'neutral', 'positive']
Nothing in Fackrells discovery of an outstanding warrant so attenuated the connection between his wrongful behavior and his detection of drugs as to diminish the exclusionary rules deterrent benefits.
The existence of a personality disorder or mental-health issue
his ability to obtain a more favorable arrangement


### Generate the sentiments of legal documents using `ibm/granite-4-h-micro` model


Get the docs summaries

In [19]:
results: list[str] = []
for sentence in data_train["Phrase"][:10]:
    messages = get_messages(example_sentences, example_sentiments, sentence)

    response = model.chat(messages)
    results.append(response["choices"][0]["message"]["content"].lower())

results

['neutral',
 'negative',
 'neutral',
 'neutral',
 'neutral',
 'negative',
 'positive',
 'neutral',
 'negative',
 'negative']

<a id="Score-the-model"></a>
## Score the model

**Note:** To run the Score section for model scoring on the whole financial phrasebank dataset, please transform following `markdown` cells to `code` cells.
Have in mind that scoring model on the whole test set can take significant amount of time.

Get the true labels

In [20]:
reverse_label_map = {v: k for k, v in label_map.items()}

y_true = [reverse_label_map[item] for item in y_test.values[:10]]
y_true

[-1, 0, -1, -1, -1, -1, -1, 0, -1, 1]

Get the predicted labels

In [21]:
y_pred = [reverse_label_map[result.strip().lower()] for result in results]
y_pred

[0, -1, 0, 0, 0, -1, 1, 0, -1, -1]

Calculate the mean squared error (MSE)

In [22]:
mse = sum(abs(pred - true) ** 2 for pred, true in zip(y_pred, y_true)) / len(y_pred)
mse

1.3

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to find sentiments of legal documents with `ibm/granite-4-h-micro` on watsonx. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 


### Authors
**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.